<a href="https://colab.research.google.com/github/tanpaterusan/antologiWISH/blob/main/Pertemuan12_Amalia_240401010284.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pertemuan XII

*   Nama: Amalia
*   NIM: 240401010284
*   Kelas: IF403

In [7]:
# Langkah 1: Generate & Eksplorasi Dataset Transaksi
# Buat dataset transaksi sintetis dengan pola pembelian tersembunyi, lalu eksplorasi frekuensi tiap produk.

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']
# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# # Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')
print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [8]:
# Langkah 2: One-Hot Encoding Transaksi
# Ubah daftar transaksi menjadi tabel one-hot encoding menggunakan TransactionEncoder.

from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [9]:
# Langkah 3: Cari Frequent Itemset dengan Apriori
# Jalankan Apriori dengan beberapa nilai min_support, amati bagaimana jumlah itemset yang ditemukan berubah.

from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')
# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Langkah 4: Bentuk & Saring Aturan Asosiasi
# Bentuk aturan asosiasi, saring dengan min_confidence dan min_lift, lalu urutkan berdasarkan Lift tertinggi.

from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)
print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))


**Interpretasi:**

*   Aturan mana yang paling kuat (dengan nilai Lift tertinggi)?
    Perhatikan baris-baris teratas dari hasil `rules` yang sudah diurutkan berdasarkan Lift. Nilai Lift di atas 1 menunjukkan adanya hubungan positif antar item. Semakin tinggi nilai Lift-nya, semakin kuat dan menarik pola hubungan tersebut.

*   Apakah aturan tersebut masuk akal secara bisnis?
    Contoh yang sering kita temui adalah 'Roti' dan 'Selai'. Jika aturannya 'Roti -> Selai' memiliki Lift tinggi, ini mengindikasikan bahwa pembeli yang membeli Roti cenderung juga membeli Selai. Pola seperti ini sangat berguna untuk penataan produk di toko, rekomendasi, atau strategi promosi. Coba perhatikan aturan-aturan lain yang muncul, apakah ada yang menarik dan bisa Anda kaitkan dengan perilaku pembelian sehari-hari?

In [12]:
# Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
# Bangun katalog produk dengan kategori, lalu buat rekomendasi produk serupa menggunakan cosine similarity atas kategori (one-hot).

from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
  print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [16]:
# Langkah 6: Bandingkan Kedua Pendekatan
# Bandingkan rekomendasi dari aturan asosiasi (Langkah 4) dengan rekomendasi Content- Based (Langkah 5) untuk produk yang sama.

from mlxtend.frequent_patterns import association_rules
rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target

rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### Diskusi Konsistensi Rekomendasi dan Pendekatan Hybrid

Setelah membandingkan rekomendasi dari **Aturan Asosiasi** dan **Content-Based Filtering** untuk produk yang sama (`Roti`), kita dapat mengamati apakah kedua pendekatan tersebut menghasilkan rekomendasi yang konsisten.

Pada kasus ini, rekomendasi dari **Aturan Asosiasi** menunjukkan bahwa pembeli 'Roti' cenderung juga membeli 'Selai', dengan nilai `lift` yang relatif tinggi. Hal ini mengindikasikan adanya pola pembelian bersama yang signifikan berdasarkan data transaksi historis.

Di sisi lain, **Content-Based Filtering** juga merekomendasikan 'Selai', serta produk lain seperti 'Sereal' dan 'Susu'. Rekomendasi 'Selai' muncul karena produk tersebut memiliki kategori yang sama dengan 'Roti' (yaitu 'Bakery'). Rekomendasi 'Sereal' dan 'Susu' juga mungkin muncul jika kategori mereka memiliki kesamaan atau dianggap relevan secara kontekstual dengan 'Roti' dalam katalog produk kita.

#### Apakah Kedua Pendekatan Memberi Rekomendasi yang Konsisten?

Dalam contoh ini, terdapat **konsistensi parsial**. Keduanya merekomendasikan 'Selai' sebagai pendamping 'Roti'. Konsistensi ini terjadi karena 'Selai' memiliki hubungan kuat secara transaksional dengan 'Roti' (ditemukan oleh aturan asosiasi) dan juga kesamaan kategori (ditemukan oleh content-based filtering). Namun, mungkin ada produk lain yang direkomendasikan oleh satu pendekatan tetapi tidak oleh yang lain, menunjukkan bahwa masing-masing metode menangkap aspek hubungan produk yang berbeda.

#### Kapan Sebaiknya Menggunakan Salah Satu, atau Menggabungkan Keduanya (Hybrid)?

1.  **Aturan Asosiasi (Item-based/Market Basket Analysis):**
    *   **Kapan digunakan:** Ideal ketika kita ingin menemukan pola pembelian yang sering terjadi secara bersamaan antar produk. Ini sangat berguna untuk penataan toko (misalnya, menempatkan produk terkait berdekatan), promosi bundling, atau rekomendasi produk tambahan (*upselling/cross-selling*) kepada pelanggan yang telah membeli produk tertentu. Kekuatannya terletak pada kemampuannya mengungkap hubungan tak terduga yang mungkin tidak terlihat dari metadata produk saja.
    *   **Kelemahan:** Membutuhkan data transaksi historis yang ekstensif dan bisa menjadi kurang efektif untuk item baru tanpa riwayat transaksi yang cukup (masalah *cold-start*).

2.  **Content-Based Filtering:**
    *   **Kapan digunakan:** Cocok ketika kita memiliki informasi atribut yang kaya tentang produk (kategori, merek, deskripsi, fitur, dll.). Pendekatan ini merekomendasikan item yang serupa dengan item yang disukai pengguna di masa lalu. Sangat berguna untuk merekomendasikan item baru atau item yang jarang dibeli karena tidak bergantung pada riwayat transaksi yang sering.
    *   **Kelemahan:** Cenderung merekomendasikan item yang sangat mirip, sehingga kurang mampu memperkenalkan variasi atau menemukan preferensi yang sama sekali baru (*serendipity*).

3.  **Pendekatan Hybrid:**
    *   **Kapan digunakan:** Seringkali merupakan solusi terbaik untuk menggabungkan keunggulan dari kedua pendekatan dan memitigasi kelemahannya. Misalnya, pendekatan hibrida dapat menggunakan Content-Based Filtering untuk mengatasi masalah *cold-start* bagi produk baru, kemudian beralih ke Aturan Asosiasi setelah produk memiliki cukup data transaksi.
    *   **Manfaat:** Meningkatkan akurasi rekomendasi, memberikan variasi yang lebih baik, dan dapat menangani kasus *cold-start* dengan lebih efektif.

**Kesimpulan:** Pemilihan pendekatan terbaik atau kombinasi keduanya sangat tergantung pada jenis data yang tersedia, tujuan bisnis, dan kompleksitas sistem yang ingin dibangun. Memahami karakteristik masing-masing metode memungkinkan kita untuk merancang sistem rekomendasi yang lebih cerdas dan efektif.